# Import dependencies

In [ ]:
import Levenshtein
import networkx as nx

from bertopic import BERTopic
from dask import dataframe as dd
from dask.distributed import Client # only for dashboard
from keybert import KeyBERT
from pyvis.network import Network
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
client = Client() # only for dashboard
print(client.dashboard_link) # only for dashboard

http://127.0.0.1:8787/status


In [ ]:
sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')
physics_in_visualization = True
concept_similarity_threshold = 0.8

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Load docs and topics

In [4]:
# Original dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv
# Kaggle is running into issues rendering the dataset page due to its README so one can use an older snapshot from Zenodo in the meantime.
# Zenodo dataset snapshot: https://zenodo.org/records/15808027
# format zenodo dataset to match what we expect from kaggle
# jq -c '.edges | map(.edge as $id | .attrs | .update_date = .date | del(.date) | .id = $id)' arxiv.json > arxiv-fixed.json

## Load BERTopic model

In [5]:
topic_model = BERTopic.load("artifacts/BERTopic/bertopic-concat-1632186.model", embedding_model=sentence_transformer)

## Get top ten topics

In [ ]:
all_topics = topic_model.get_topics()
# print(all_topics)

# keep only top 10 topics and skip the -1 outlier topic
topics = {k: v for k, v in all_topics.items() if k < 10 and k != -1}
print(topics)

# get terms for each topic
topic_terms = {k: [term[0] for term in v] for k, v in topics.items()}
print(topic_terms)

{0: [('stars', np.float64(0.01630549608019157)), ('galaxies', np.float64(0.01585385039307181)), ('dark', np.float64(0.015176884332386046)), ('stellar', np.float64(0.013666200949412658)), ('galaxy', np.float64(0.013192773278642048)), ('emission', np.float64(0.012485501442077376)), ('gravitational', np.float64(0.012254120556497375)), ('black hole', np.float64(0.01194106536073987)), ('hole', np.float64(0.011748812431097107)), ('dark matter', np.float64(0.011242056733539747))], 1: [('algebras', np.float64(0.026076028714009523)), ('varieties', np.float64(0.013982209547931909)), ('cohomology', np.float64(0.01332090479171911)), ('polynomials', np.float64(0.013196135718240767)), ('elliptic', np.float64(0.01218694209356435)), ('projective', np.float64(0.012076294229271296)), ('abelian', np.float64(0.009891304285245293)), ('fractional', np.float64(0.009677183519498806)), ('moduli', np.float64(0.009566441883420398)), ('invariants', np.float64(0.009341290182796119))], 2: [('wireless', np.float64(0

## Load Arxiv corpus

In [ ]:
df = dd.read_parquet(f"artifacts/Preprocessing/arxiv-metadata-cleaned_BERTopic.parquet")
df["corpus"] = df["title"] + ". " + df["abstract"]

## Compute corpus as a dict of id to concatenated title and abstract

In [8]:
corpus = (
    df[["id", "corpus"]]
    .head(10000)
    # .compute()
    .set_index("id")["corpus"]
    .to_dict()
)

# print(corpus.keys())
# print(corpus["2501.15274"])
# print(corpus["2404.07486"])

/usr/local/lib/python3.13/dist-packages/dask/dataframe/core.py:383: UserWarning: Insufficient elements for `head`. 10000 elements requested, only 6446 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(


## Get document-topic assignments

In [9]:
document_topics, probs = topic_model.transform(list(corpus.values()))

Batches:   0%|          | 0/202 [00:00<?, ?it/s]

2026-09-11 10:31:36,452 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-09-11 10:31:42,686 - BERTopic - Dimensionality - Completed ✓
2026-09-11 10:31:42,687 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-09-11 10:31:48,910 - BERTopic - Cluster - Completed ✓


In [10]:
# print(document_topics)
# print(topics)
for i in range(10):
    count = document_topics.count(i)
    print(f"Topic {i}: {count} documents")

Topic 0: 1054 documents
Topic 1: 624 documents
Topic 2: 155 documents
Topic 3: 124 documents
Topic 4: 101 documents
Topic 5: 91 documents
Topic 6: 91 documents
Topic 7: 98 documents
Topic 8: 98 documents
Topic 9: 91 documents


In [12]:
# pull documents for topics 0-9 from topics from the corpus and create a dictionary of documents by topics
documents_by_topic = {}
all_ids = list(corpus.keys())
for i, doc_topic in enumerate(document_topics):
    doc_topic = int(doc_topic)
    if doc_topic < 10 and doc_topic != -1:
        if doc_topic not in documents_by_topic:
            documents_by_topic[doc_topic] = {}
        document_id = all_ids[i]
        documents_by_topic[doc_topic][document_id] = corpus[document_id]

In [13]:
print(documents_by_topic.keys())
print(list(documents_by_topic[0].keys())[:5])
print(documents_by_topic[0]['2004.10097'])

dict_keys([7, 0, 1, 8, 3, 4, 9, 2, 6, 5])
['2004.10097', '2203.06878', '2503.14025', '2004.07263', '2003.08845']
the spin-temperature dependence of the 21cm -- lae cross-correlation. cross-correlating 21cm with known cosmic signals will be invaluable proof of the cosmic origin of the first 21cm detections. as some of the widest fields available, comprising thousands of sources with reasonably known redshifts, narrow-band lyman alpha emitter (lae) surveys are an obvious choice for such cross-correlation. here we revisit the 21cm -- lae cross-correlation, relaxing the common assumption of reionization occurring in a pre-heated intergalactic medium (igm). using specifications from the square kilometre array and the subary hyper supreme-cam, we present new forecasts of the 21cm -- lae cross-correlation function at . we sample a broad parameter space of the mean igm neutral fraction and spin temperature, (, ). the sign of the cross-correlation roughly follows the sign of the 21cm signal: io

# Extract concepts with KeyBERT

## Init KeyBERT

In [14]:
keybert_model = KeyBERT()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Extract keyphrases (concepts) for each document

In [ ]:
concepts_per_document_per_topic = {}
for topic, docs in documents_by_topic.items():
    print(topic)
    concepts_per_document_per_topic[topic] = {}
    c = 0
    for doc_id, doc_content in docs.items():
        print(f"  {c}", end='\r')
        # doc_title, doc_content = doc.split(": ", maxsplit=1)  # Split title from content
        keyphrases = keybert_model.extract_keywords(doc_content, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=5)
        concepts_per_document_per_topic[topic][doc_id] = [kw[0] for kw in keyphrases]  # Store only the keyphrases
        c += 1
        if c >= 5: # limit to 5 documents per topic for now
            print()
            break

7
  4
0
  4
1
  4
8
  4
3
  4
4
  4
9
  4
2
  4
6
  4
5
  4


In [16]:
print(concepts_per_document_per_topic)

{7: {'2404.07486': ['graphs chromatic', 'chromatic number', 'result chromatic', 'free graphs', 'free graph'], '2107.10105': ['fullerene graphs', 'graphs fullerene', 'fullerene graph', 'complexity fullerene', 'models fullerene'], '2412.09410': ['acyclically colourable', 'list colourable', 'graphs acyclically', 'colourable bound', 'vertex colouring'], '2103.11892': ['free colorings', 'colorings vertex', 'edge colorings', 'vertices colors', 'graphs unique'], '2112.04230': ['isospectral graphs', 'graphs isospectral', 'isospectral metric', 'isospectrality', 'metric graphs']}, 0: {'2004.10097': ['heated intergalactic', 'cosmic signals', 'bias cosmic', 'intergalactic', 'known cosmic'], '2203.06878': ['intrinsic nucleon', 'vacuum weinberg', 'nucleon vertex', 'qcd', 'pion nucleon'], '2503.14025': ['cosmic rays', 'propagation heliosphere', 'cosmic protons', 'modulation cosmic', 'cosmic ray'], '2004.07263': ['cosmological models', 'cosmological parameter', 'universe modifications', 'cosmology', '

# Build knowledge graph

## Initialize the graph

In [17]:
G = nx.MultiDiGraph()

## Add nodes and edges for topics and their top terms
keybert concept -> is mentioned in -> doc -> belongs_to -> topic -> is_described_by -> top terms

topic -> is_about -> keybert concept

if keybert concept already exists as topic top term: concept -> mentions -> doc; concept -> matches/is_same_as -> top term

In [ ]:
for topic in topics:
    # Add topic nodes
    if topic not in G.nodes:
        G.add_node(topic, label=f"{topic}", type="topic", physics=physics_in_visualization)
    
    # Add top term nodes
    top_terms = topic_terms[topic]
    for term in top_terms:
        if term not in G.nodes:
            G.add_node(term, label=f"{term}", type="top_term", physics=physics_in_visualization)
        G.add_edge(topic, term, relation="is_described_by", physics=physics_in_visualization)


## Add nodes and edges for document concepts

In [19]:
all_concepts = []

for topic, docs in concepts_per_document_per_topic.items():
    for doc_id, doc_concepts in docs.items():
        # add doc node
        G.add_node(doc_id, label=doc_id, type="document", physics=physics_in_visualization)
        # link doc to topic
        G.add_edge(doc_id, topic, relation="belongs_to", physics=physics_in_visualization)
        for concept in doc_concepts:
            all_concepts.append(concept)
            if concept in G.nodes:
                # concept already exists as node, check if node is of type top_term
                if G.nodes[concept]['type'] == 'top_term':
                    top_term = concept
                    concept = f"{concept} concept"
                    G.add_node(concept, label=f"{top_term}", type="concept", physics=physics_in_visualization)
                    print(f"Concept {top_term} already exists as top term, linking...")
                    G.add_edge(concept, top_term, relation="is_same_as", physics=physics_in_visualization)
            else:
                #add concept node
                G.add_node(concept, label=f"{concept}", type="concept", physics=physics_in_visualization)
                all_concepts.append(concept)
            # add edge from concept to doc
            G.add_edge(doc_id, concept, relation="mentions", physics=physics_in_visualization)
            # add edge from concept to topic
            G.add_edge(concept, topic, relation="is_about", physics=physics_in_visualization)

Concept chromatic number already exists as top term, linking...
Concept galaxies already exists as top term, linking...
Concept point cloud already exists as top term, linking...
Concept reinforcement learning already exists as top term, linking...
Concept segmentation already exists as top term, linking...


## Compute concept similarity matrix

In [20]:
concept_embeddings = sentence_transformer.encode(all_concepts, convert_to_tensor=True)
similarity_matrix = cosine_similarity(concept_embeddings)

## Add concept similarity edges

In [ ]:
for i in range(len(all_concepts)):
    for j in range(i+1, len(all_concepts)):
        if Levenshtein.distance(all_concepts[i], all_concepts[j]) > 3:  # only consider concepts that are not too similar in spelling
            if similarity_matrix[i, j] > concept_similarity_threshold:
                a = all_concepts[i]
                b = all_concepts[j]
                if a not in G.nodes:
                    a = a + " concept"
                if b not in G.nodes:
                    b = b + " concept"
                G.add_edge(a, b, relation="is_related_to")

## Add legend nodes indicating what a document, topic, and concept node is

In [22]:
# legend_nodes = [
#     ("Topic Term Node", "Topic Term Node", "top_term", -700, -400),
#     ("Topic Node", "Topic Node", "topic", -700, -300),
#     ("Document Node", "Document Node", "document", -700, -200),
#     ("Concept Node", "Concept Node", "concept", -700, -100),
# ]

# for node_id, label, node_type, x, y in legend_nodes:
#     G.add_node(
#         node_id,
#         label=label,
#         type=node_type,
#         x=x,
#         y=y,
#         # fixed=True,
#         physics=False,
#         size=15,
#     )

# Visualize knowledge graph

## Init pyvis Network

In [26]:
net = Network(height="1080px", select_menu=True, filter_menu=True, notebook=True, cdn_resources="remote")
net.from_nx(G)

## Customize node appearance based on types

In [27]:
for node in G.nodes(data=True):
    node_id = node[0]
    node_type = node[1].get("type")
    if node_type == "top_term":
        color = "purple"
        shape = "star"
    elif node_type == "topic":
        color = "green"
        shape = "diamond"
    elif node_type == "document":
        color = "lightblue"
        shape = "circle"
    else:
        color = "orange"
        shape = "square"
    net.get_node(node_id)["color"] = color
    net.get_node(node_id)["shape"] = shape

## Output to HTML

In [28]:
net.show("knowledge_graph.html")

knowledge_graph.html
